# Qwen + original Pre-commit Hook: one-run integration

This notebook requires a trusted local POSIX filesystem shared with a Docker-capable host. It is **not yet a remote Colab-to-Docker connector**. Use a notebook kernel on the Docker host (GPU recommended). Do not execute generated shell commands in this kernel. Original task prompts are preserved; no reasoning-elicitation instruction is added. Read `docs/QWEN_INTEGRATION.md` first. No cells have been run in the delivered notebook.

In [ ]:
from pathlib import Path
import json, sys, subprocess, uuid
ROOT = Path.cwd().resolve()
assert (ROOT / 'scripts/run_precommit_qwen.py').is_file(), 'Start the notebook from the capstone repository root'
sys.path.insert(0, str(ROOT / 'src'))
subprocess.run([sys.executable, 'scripts/verify_upstream.py'], cwd=ROOT, check=True)
subprocess.run(['docker', 'info'], check=True, capture_output=True)


Install the model dependencies in the notebook environment before proceeding: `python -m pip install -r requirements.txt`. Use Python 3.11 or 3.12. The container builds its own original dependencies. This cell prepares files only; it does not start a container or load weights.

In [ ]:
run_dir = ROOT / 'results' / ('precommit_' + uuid.uuid4().hex)
subprocess.run([sys.executable, 'scripts/run_precommit_qwen.py', '--run-dir', str(run_dir), '--prepare-only'], cwd=ROOT, check=True)
config = json.loads((run_dir / 'config.json').read_text())
launch = json.loads((run_dir / 'launch.json').read_text())
print(json.dumps(config, indent=2))


The next cell loads Qwen and starts inference only. It does not run shell actions. Model weights may download. Use a pinned model revision for research runs; the resolved commit is saved in each generation record.

In [ ]:
from qwen_adapter.settings import validate
from qwen_adapter.local_model import LocalQwen
from qwen_adapter.worker import serve
import threading
validate(config)
engine = LocalQwen(config['agent']['model'], config['agent']['qwen_options']['generation'])
stop_worker = threading.Event()
worker = threading.Thread(target=serve, args=(run_dir / 'bridge', engine, stop_worker), daemon=True)
worker.start()


Build and run the CPU task container using the prepared fixed launch arguments. The original task entrypoint executes inside Docker. No host directories other than this run’s bridge/config/output are mounted. Do not run two containers for the same run. This blocking cell has a timeout and removes the container on timeout.

In [ ]:
subprocess.run(launch['commands'][0], check=True)
try:
    with (run_dir / 'controller.log').open('w') as log:
        completed = subprocess.run(launch['commands'][1], stdout=log, stderr=subprocess.STDOUT, timeout=3600)
    print('Controller exit:', completed.returncode)
except subprocess.TimeoutExpired:
    command = launch['commands'][1]
    container_name = command[command.index('--name') + 1]
    subprocess.run(['docker', 'rm', '-f', container_name], check=False, capture_output=True)
    raise
finally:
    stop_worker.set()
    worker.join(timeout=5)


In [ ]:
print((run_dir / 'controller.log').read_text()[-12000:])
for path in sorted((run_dir / 'bridge').glob('*.response.json')):
    response = json.loads(path.read_text())
    print(path.name, response.get('parsed', {}).get('kind'), response.get('error'))
print('Saved:', run_dir)


Inspect every raw generation and tool result before annotation. Pre-tool text is only an observable explanation; missing reasoning is not a negative deception label. No tool call means final report under original mode, and this final turn has no separately established reasoning/report boundary. The trusted container wrapper exports checkpoint ownership to the notebook user on normal exit; forced termination may require manual recovery. Keep outputs on persistent POSIX storage or export after completion. Do not automatically restart interrupted runs.